# 04 · Validación de Datos
Verifica integridad, conteos y calidad básica de cada tabla.

In [ ]:
from google.cloud import bigquery
from dotenv import load_dotenv
import os, pandas as pd

load_dotenv()
os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = os.getenv('GOOGLE_APPLICATION_CREDENTIALS')
PROJECT_ID = os.getenv('GCP_PROJECT_ID')
DATASET_ID = os.getenv('BQ_DATASET_ID')
client = bigquery.Client(project=PROJECT_ID)
DS = f'{PROJECT_ID}.{DATASET_ID}'

In [ ]:
tables = ['categories','customers','products','orders','order_item_id','payments','reviews']
rows = []
for t in tables:
    n = client.query(f'SELECT COUNT(*) as n FROM `{DS}.{t}`').result().to_dataframe().iloc[0]['n']
    rows.append({'tabla': t, 'filas': n})
pd.DataFrame(rows)

In [ ]:
q = f'SELECT status, COUNT(*) AS total FROM `{DS}.orders` GROUP BY status ORDER BY total DESC'
client.query(q).result().to_dataframe()

In [ ]:
q = f"""
SELECT COUNT(DISTINCT o.order_id) AS num_orders,
       ROUND(SUM(oi.unit_price * oi.quantity * (1 - oi.discount)), 2) AS revenue_total,
       ROUND(AVG(oi.unit_price * oi.quantity * (1 - oi.discount)), 2) AS avg_item_value
FROM `{DS}.orders` o JOIN `{DS}.order_item_id` oi USING (order_id)
WHERE o.status = 'delivered'"""
client.query(q).result().to_dataframe()

In [ ]:
q = f"""
SELECT p.name, SUM(oi.quantity) AS units_sold,
       ROUND(SUM(oi.unit_price * oi.quantity * (1-oi.discount)),2) AS revenue
FROM `{DS}.order_item_id` oi JOIN `{DS}.products` p USING (product_id)
JOIN `{DS}.orders` o USING (order_id)
WHERE o.status = 'delivered'
GROUP BY p.name ORDER BY revenue DESC LIMIT 5"""
client.query(q).result().to_dataframe()

In [ ]:
q = f'SELECT rating, COUNT(*) AS total FROM `{DS}.reviews` GROUP BY rating ORDER BY rating DESC'
client.query(q).result().to_dataframe()

In [ ]:
q = f'SELECT acquisition_channel, COUNT(*) AS clientes FROM `{DS}.customers` GROUP BY 1 ORDER BY 2 DESC'
client.query(q).result().to_dataframe()

In [ ]:
q = f'SELECT method, status, COUNT(*) AS n, ROUND(SUM(amount),2) AS total_amount FROM `{DS}.payments` GROUP BY 1,2 ORDER BY 1,3 DESC'
client.query(q).result().to_dataframe()